## Initialization

In [1]:
from active_learning.al_HER_sulfide import AL
from utils.utils import get_tasks

# exp selection
# exp_name = 'acidic_mor_PtRuSc'
exp_name = 'Ali_HER_sulfide'
# exp_name = 'DFFC_PdPtCu'
#exp_name = 'DFFC_8D_4'

## Save and load

In [2]:
# load al instance from database
al = AL(exp_name)

In [11]:
# save the current al instance to the database
al.save_exp_to_db()

## AL main workflow

### recipe generation

In [3]:
%%time
# generate next trial using Bayesian Optimization, specify beta if needed, e.g. beta=100 for exploration, beta=0.2 for default
al.generate_botorch_trial(n=30, aquisition = 'qUCB', optimizer_kwargs = {'options':{'maxiter':20000000}, 'sequential':True})

BO trial successfully added as trial 1
CPU times: total: 14min 52s
Wall time: 6min 18s


In [1]:
# generate next trial using SOBOL method (random select)
al.generate_sobol_trial(n=20)

NameError: name 'al' is not defined

In [6]:
# update the recipe table on the database, if trial index is not specified, the recipe of the latest trial will be uploaded
al.update_recipe_table(trial_index=None, add_pred=False)

### sample preparation

In [7]:
# benchmark specification
benchmark = None
# benchmark={5: '1_0'}

In [ ]:
# command opentrons to prepare the sample for the latest trial, following the ot2_run_configs of the exp
al.make_sample(trial_index=None, benchmark=benchmark, operator_name='chu')

In [8]:
# update the sample table on the database, if trial index is not specified, the sample of the latest trial will be uploaded. Note that sample table calculation is based on local ot2_run_configs.py under project folder, not the one on ot2 server.
al.update_sample_table(trial_index=None, benchmark=benchmark)

### sample testing complete

In [9]:
# run this cell after sample testing and data analysis is completed, then go back to the BO step
al.mark_trial_complete(trial_index=None)

BatchTrial(experiment_name='Ali_HER_sulfide', index=0, status=TrialStatus.RUNNING)


## AL monitor

In [ ]:
# update the exp with the latest result
al.update_exp_result()

In [4]:
# exp monitor for windows
al.exp_monitor()

,arm_name,trial_index,arm_index,Pt,Ni,Mo,Co,Cu,Fe,pred_mean,pred_std,potential,trial_status,generation_method
0,0_0,0,0,0.750,0.250,0.000,0.000,0.000,0.000,-2.039,0.04107,-2.07,COMPLETED,Manual
1,0_1,0,1,0.750,0.000,0.250,0.000,0.000,0.000,-2.044,0.03517,-2.03,COMPLETED,Manual
2,0_2,0,2,0.750,0.000,0.000,0.250,0.000,0.000,-2.033,0.03697,-2.06,COMPLETED,Manual
3,0_3,0,3,0.750,0.000,0.000,0.000,0.250,0.000,-2.051,0.03707,-2.04,COMPLETED,Manual
4,0_4,0,4,0.750,0.125,0.125,0.000,0.000,0.000,-1.983,0.03400,-2.00,COMPLETED,Manual
5,0_5,0,5,0.750,0.125,0.000,0.125,0.000,0.000,-1.957,0.03389,-1.91,COMPLETED,Manual
6,0_6,0,6,0.750,0.000,0.125,0.125,0.000,0.000,-2.036,0.03011,-2.00,COMPLETED,Manual
7,0_7,0,7,0.750,0.125,0.000,0.000,0.125,0.000,-1.967,0.03398,-1.93,COMPLETED,Manual
8,0_8,0,8,0.750,0.000,0.125,0.000,0.125,0.000,-2.060,0.03021,-2.13,COMPLETED,Manual
9,1_0,1,0,0.647,0.131,0.000,0.166,0.056,0.000,-1.958,0.03836,NaN,RUNNING,BoTorch


In [4]:
# exp monitor for mac and linux
from IPython.display import display, HTML
display(HTML(al.exp_monitor().to_html()))

,arm_name,trial_index,arm_index,Pt,Ni,Mo,Co,Cu,Fe,pred_mean,pred_std,potential,trial_status,generation_method
0,0_0,0,0,0.750,0.250,0.000,0.000,0.000,0.000,-2.039,0.04107,-2.07,COMPLETED,Manual
1,0_1,0,1,0.750,0.000,0.250,0.000,0.000,0.000,-2.044,0.03517,-2.03,COMPLETED,Manual
2,0_2,0,2,0.750,0.000,0.000,0.250,0.000,0.000,-2.033,0.03697,-2.06,COMPLETED,Manual
3,0_3,0,3,0.750,0.000,0.000,0.000,0.250,0.000,-2.051,0.03707,-2.04,COMPLETED,Manual
4,0_4,0,4,0.750,0.125,0.125,0.000,0.000,0.000,-1.983,0.03400,-2.00,COMPLETED,Manual
5,0_5,0,5,0.750,0.125,0.000,0.125,0.000,0.000,-1.957,0.03389,-1.91,COMPLETED,Manual
6,0_6,0,6,0.750,0.000,0.125,0.125,0.000,0.000,-2.036,0.03011,-2.00,COMPLETED,Manual
7,0_7,0,7,0.750,0.125,0.000,0.000,0.125,0.000,-1.967,0.03398,-1.93,COMPLETED,Manual
8,0_8,0,8,0.750,0.000,0.125,0.000,0.125,0.000,-2.060,0.03021,-2.13,COMPLETED,Manual
9,1_0,1,0,0.647,0.131,0.000,0.166,0.056,0.000,-1.958,0.03836,NaN,RUNNING,BoTorch


In [ ]:
# check abandon reason
al.exp.trials[0].abandoned_reason

## AL manual edit

In [ ]:
# update the exp with the latest result and force match previous trials with the results in the database
al.update_exp_result(force_update_trial=[0, 1])

In [5]:
# manually add a trial
trial_df = get_tasks(exp_name, task_name='manual_1')
al.add_manual_trial(trial_df)

    Ni   Mo   Co   Cu   Fe   Pt
0  1.0  0.0  0.0  0.0  0.0  3.0
1  0.0  1.0  0.0  0.0  0.0  3.0
2  0.0  0.0  1.0  0.0  0.0  3.0
3  0.0  0.0  0.0  1.0  0.0  3.0
4  0.5  0.5  0.0  0.0  0.0  3.0
5  0.5  0.0  0.5  0.0  0.0  3.0
6  0.0  0.5  0.5  0.0  0.0  3.0
7  0.5  0.0  0.0  0.5  0.0  3.0
8  0.0  0.5  0.0  0.5  0.0  3.0

Please double check task list as above
manual trial successfully added as trial 0


In [ ]:
# abandon an certain arm in a trial
al.abandon_arm(0, '0_18', 'abandoned in manual test')

In [ ]:
# abandon a whole trial
al.abandon_trial(9, 'test trial')

## Plotting

In [ ]:
from ax.modelbridge.cross_validation import cross_validate
from ax.plot.contour import interact_contour
from ax.plot.diagnostic import interact_cross_validation
from ax.plot.scatter import interact_fitted
from ax.plot.slice import interact_slice
from ax.utils.notebook.plotting import render, init_notebook_plotting
init_notebook_plotting()

In [ ]:
model = al.get_bo_model()

In [ ]:
# Contour plots
render(interact_contour(model=model, metric_name='max_power'))

In [ ]:
# Cross-validation plots
cv_results = cross_validate(model)
print('current r_squared is: ', al.calculate_r_squared(cv_results))
render(interact_cross_validation(cv_results))

In [ ]:
# Slice plots
render(interact_slice(model))

In [ ]:
# Tile plots
render(interact_fitted(model, rel=False))

## Danger Zone

In [ ]:
# delete the current active learning process from the database
al.delete_exp_on_db()

In [2]:
# only use this if you want to start a new experiment, note that the previous experiment has to be manually deleted, first, because the script refuses to overwrite the previous experiment by default
al = AL(exp_name, load_from_db=False)

Database Ali_HER_sulfide does not exist yet, creating now...
Table recipe does not exist on server yet, creating now...
Table sample does not exist on server yet, creating now...
Table performance does not exist on server yet, creating now...
full table view created
